# M0 · 08 — Autograd & a tiny training loop

This ties everything together. PyTorch **autograd** computes gradients for you;
an **optimizer** nudges parameters to reduce the loss. The exact 5-line loop you
see in the GPT:

```python
optimizer.zero_grad(set_to_none=True)
loss.backward()
optimizer.step()
```

We'll fit a trivial problem so you can *see* the loss go down.

In [ ]:
import torch
from m0_checks import check, check_tensor, TODO
torch.manual_seed(0)

## 1. `requires_grad` — track operations for differentiation

A tensor with `requires_grad=True` remembers how it was used so gradients can flow
back to it. Create `w = 5.0` as a **float tensor that requires grad**.

In [ ]:
w = TODO
w

In [ ]:
check('requires grad', bool(w.requires_grad) if w is not TODO else TODO, True)

## 2. `.backward()` computes the gradient

Let `loss = w**2`. Calculus says `d(loss)/dw = 2*w = 10` at `w=5`. Call
`loss.backward()` and read `w.grad` — autograd should give exactly 10.

In [ ]:
loss = w ** 2
# YOUR CODE: call backward on loss
TODO
w.grad

In [ ]:
check('gradient is 2w = 10', w.grad, torch.tensor(10.0))

## 3. Gradient descent by hand — one step

To *reduce* loss, step **against** the gradient: `w := w - lr * w.grad`.
With `lr=0.1`, new `w = 5 - 0.1*10 = 4.0`. Fill in the update (inside
`torch.no_grad()` so the update itself isn't tracked).

In [ ]:
lr = 0.1
with torch.no_grad():
    new_w = TODO
float(new_w)

In [ ]:
check('one GD step -> 4.0', float(new_w), 4.0)

## 4. Why `zero_grad` — gradients accumulate

PyTorch *adds* new gradients onto existing ones. If you forget to clear them, they
pile up and corrupt training. Run this to see it: calling `backward()` twice
without zeroing doubles the grad.

In [ ]:
a = torch.tensor(3.0, requires_grad=True)
(a * 2).backward()
print('after 1st backward, a.grad =', a.grad.item())  # 2
(a * 2).backward()
print('after 2nd backward, a.grad =', a.grad.item())  # 4 (accumulated!)
check('grads accumulated to 4', a.grad, torch.tensor(4.0))

## 5. The full loop — fit `y = 3x` with an optimizer

Now the real pattern. We have data where the true relationship is `y = 3x`, and a
single parameter `w` (starting wrong). Complete the **three lines** of the loop so
`w` converges toward 3 and the loss drops toward 0.

Order: `zero_grad` → `backward` → `step`.

In [ ]:
import torch
torch.manual_seed(0)
x = torch.tensor([1., 2., 3., 4.])
y = 3 * x                       # true relationship
w = torch.tensor(0.0, requires_grad=True)
optimizer = torch.optim.SGD([w], lr=0.01)

for step in range(200):
    pred = w * x
    loss = ((pred - y) ** 2).mean()
    # --- YOUR CODE: the 3-line update, in the right order ---
    TODO
    # --------------------------------------------------------

print('final w =', round(w.item(), 3), ' (target 3.0)')
print('final loss =', round(loss.item(), 6))

In [ ]:
check('w converged to ~3', round(w.item()), 3)
check('loss near zero', loss.item() < 1e-2, True)

## ✅ Recap — you now understand the GPT training loop

- `requires_grad=True` tracks a tensor for differentiation.
- `loss.backward()` fills `.grad` with `d(loss)/d(param)`.
- `optimizer.step()` nudges params against the gradient; `zero_grad()` clears the
  accumulated grads first.
- The GPT's loop is *identical* — just with millions of parameters and
  `F.cross_entropy` as the loss.

🎉 **M0 complete.** You can now read every line of `M2/single_self_attention.py`.